### This notebook shows the use cases of the Runner class on a trivial example.

In [1]:
# import the Runner and RunnerViewer classes from the empirical package
from empirical import Runner
from empirical.viewer import RunnerViewer

# other imports 
import numpy as np

import time

Please read the `experiment_example.ipynb` file first.

It is common to test multiple configurations for the following function:

In [2]:
def numerical_experiment(num_iterations, decay_rate):
    steps = np.arange(num_iterations)
    noise_1 = np.random.normal(0, 0.2, size=num_iterations)
    noise_2 = np.random.normal(0, 0.5, size=num_iterations)
    
    error_1_vec = np.exp(-decay_rate * steps) + noise_1
    error_2_vec = 1.5 * np.exp(-decay_rate * steps) + noise_2
    
    result = []
    
    for i in range(num_iterations):
        
        # some heavyweight compute
        start_time = time.perf_counter()
        
        time.sleep(np.abs(noise_1[i]))
        result.append(error_1_vec[i] + error_2_vec[i])
        
        end_time = time.perf_counter()
        
        # various prints, e.g.,
        print(f' -> [Iteration {i + 1}] Error1: {error_1_vec[i]:.4f}   |   Error2: {error_2_vec[i]:.4f}        Time: {end_time - start_time}') 
        
    return result

In [3]:
num_experiments = 3

num_iterations_list = [10, 20, 30]
decay_rate_list = [0.75, 1.5, 2.3]

for i in range(num_experiments):
    
    print(f'\n + Experiment {i + 1}')
    
    exp_start_time = time.perf_counter()
    numerical_experiment(num_iterations_list[i], decay_rate_list[i])
    exp_end_time = time.perf_counter()
    
    print(f' - Done in {(exp_end_time - exp_start_time):.4f}s')


 + Experiment 1
 -> [Iteration 1] Error1: 0.9643   |   Error2: 1.3987        Time: 0.04074687464162707
 -> [Iteration 2] Error1: 0.3849   |   Error2: 0.9048        Time: 0.09248266695067286
 -> [Iteration 3] Error1: 0.1651   |   Error2: 0.6505        Time: 0.06305137509480119
 -> [Iteration 4] Error1: 0.0090   |   Error2: 0.9818        Time: 0.10142245888710022
 -> [Iteration 5] Error1: -0.3035   |   Error2: -0.2852        Time: 0.35834266571328044
 -> [Iteration 6] Error1: -0.2760   |   Error2: 0.4980        Time: 0.30141241615638137
 -> [Iteration 7] Error1: 0.0803   |   Error2: 0.3794        Time: 0.07426024973392487
 -> [Iteration 8] Error1: -0.0208   |   Error2: 0.1926        Time: 0.03008254198357463
 -> [Iteration 9] Error1: 0.2541   |   Error2: -0.2180        Time: 0.256692708004266
 -> [Iteration 10] Error1: -0.2818   |   Error2: 0.0585        Time: 0.2880629161372781
 - Done in 1.6086s

 + Experiment 2
 -> [Iteration 1] Error1: 0.9643   |   Error2: 1.4806        Time: 0.0373

For reasons previously stated, the Runner class comes in handy when dealing with a suite of experiments. First define your function as you would do for the Experiment decorator, but without applying the decorator this time:

In [4]:
def numerical_experiment(num_iterations, decay_rate):
    steps = np.arange(num_iterations)
    noise_1 = np.random.normal(0, 0.2, size=num_iterations)
    noise_2 = np.random.normal(0, 0.5, size=num_iterations)
    
    error_1_vec = np.exp(-decay_rate * steps) + noise_1
    error_2_vec = 1.5 * np.exp(-decay_rate * steps) + noise_2
    
    result = []
    
    for i in range(num_iterations):
        
        # some heavyweight compute
        time.sleep(np.abs(noise_1[i]))

        result.append(error_1_vec[i] + error_2_vec[i])
        
        # yield the intermediary parameters instead
        yield{
            "error_1": error_1_vec[i],
            "error_2": error_2_vec[i]
        }
        
    return result

Then define your parameter grid:

In [5]:
# required format: [{'param_1_1': val_1_1, 'param_1_2': val_1_2, ...}, {'param_2_1': val_2_1, 'param_2_2': val_2_2, ...}, ...]
params_grid = [
    {'num_iterations': 10, 'decay_rate': 0.75},
    {'num_iterations': 20, 'decay_rate': 1.5},
    {'num_iterations': 30, 'decay_rate': 2.3},
]

Next, instantiate the Runner class and run the experiments.

In [6]:
runner = Runner(suite_name = "Numerical_Experiments")

results = runner.run_grid(
        func=numerical_experiment, 
        param_grid=params_grid, 
        stop_on_error=False
    )


[2026-08-20 21:51:04] Starting suite: 'Numerical_Experiments' | Total configs: 3

[RUN] run_000_f6ce5ff2 | Params: {'num_iterations': 10, 'decay_rate': 0.75}
Starting experiment 'Numerical_Experiments'. Saved in 'results/Numerical_Experiments/20260820_215104_cd04743/run_000_f6ce5ff2'
[Wrapper] [Iteration 0] -> Time 0.0228s | error_1: 1.0181 | error_2: 0.9830
[Wrapper] [Iteration 1] -> Time 0.0162s | error_1: 0.4582 | error_2: 0.7125
[Wrapper] [Iteration 2] -> Time 0.0963s | error_1: 0.3144 | error_2: -0.8565
[Wrapper] [Iteration 3] -> Time 0.0495s | error_1: 0.1498 | error_2: -0.3485
[Wrapper] [Iteration 4] -> Time 0.1225s | error_1: -0.0683 | error_2: -0.2706
[Wrapper] [Iteration 5] -> Time 0.0591s | error_1: -0.0305 | error_2: 0.0507
[Wrapper] [Iteration 6] -> Time 0.2013s | error_1: 0.2073 | error_2: 0.5024
[Wrapper] [Iteration 7] -> Time 0.3815s | error_1: 0.3827 | error_2: 0.0099
[Wrapper] [Iteration 8] -> Time 0.1574s | error_1: -0.1499 | error_2: 0.2966
[Wrapper] [Iteration 9] 

Again, a dedicated viewer is used for inspection. This class has a list of ExperimentViewer objects, as well as additional suite-level features.

In [7]:
viewer = RunnerViewer("results/Numerical_Experiments")

In [8]:
viewer.summary()


------------------------------
SUITE: 20260820_215104_cd04743
 -> start date: 2026-08-20 21:51:04
 -> total duration: 00:00:09
 -> status breakdown: 3 SUCCESS / 0 FAILED
------------------------------



In [9]:
viewer.full_summary()


------------------------------
SUITE: 20260820_215104_cd04743
 -> start date: 2026-08-20 21:51:04
 -> total duration: 00:00:09
 -> status breakdown: 3 SUCCESS / 0 FAILED
------------------------------


 + [Experiment #0]


 ---------------------------- EXPERIMENT: 20260820_215104_cd04743 / run_000_f6ce5ff2 -----------------------------------------
 -> Start date: 2026-08-20 21:51:04 - End date: 2026-08-20 21:51:05  |  Status: SUCCESS
 -> Time: 00:00:01 (1.1899s)
 -> Peak RAM Usage: 0.02 MB
 -> Git Commit Hash: cd04743 (Uncommitted changes: True)

Parameters:
 -> num_iterations = 10
 -> decay_rate = 0.75

 + [Experiment #1]


 ---------------------------- EXPERIMENT: 20260820_215104_cd04743 / run_001_dd665471 -----------------------------------------
 -> Start date: 2026-08-20 21:51:05 - End date: 2026-08-20 21:51:09  |  Status: SUCCESS
 -> Time: 00:00:04 (4.0214s)
 -> Peak RAM Usage: 0.03 MB
 -> Git Commit Hash: cd04743 (Uncommitted changes: True)

Parameters:
 -> num_iterations = 20

In [10]:
viewer.runs[0].summary() # and so on



 ---------------------------- EXPERIMENT: 20260820_215104_cd04743 / run_000_f6ce5ff2 -----------------------------------------
 -> Start date: 2026-08-20 21:51:04 - End date: 2026-08-20 21:51:05  |  Status: SUCCESS
 -> Time: 00:00:01 (1.1899s)
 -> Peak RAM Usage: 0.02 MB
 -> Git Commit Hash: cd04743 (Uncommitted changes: True)

Parameters:
 -> num_iterations = 10
 -> decay_rate = 0.75


In [11]:
viewer.restore_code_state()

Cloning repository to: /Users/tudorpistol/Empirical/examples/results/Numerical_Experiments/20260820_215104_cd04743/restored_code...
Checking out commit: cd04743...
Applying patch: suite_uncommitted_20260820_215104.patch...
Commit and patch applied successfully.

 Code restored in:
/Users/tudorpistol/Empirical/examples/results/Numerical_Experiments/20260820_215104_cd04743/restored_code


The Runner class hashes the input configurations. If an experiment fails, it can continue executing the suite, skipping the failed cases.

In [12]:
params_grid = [
    {'num_iterations': 10, 'decay_rate': 0.75},
    {'num_iterations': -20, 'decay_rate': 1.5}, # will crash
    {'num_iterations': 30, 'decay_rate': 2.3},
]

runner = Runner(suite_name = "Numerical_Experiments")

results = runner.run_grid(
        func=numerical_experiment, 
        param_grid=params_grid, 
        stop_on_error=False
    )


[2026-08-20 21:52:01] Starting suite: 'Numerical_Experiments' | Total configs: 3

[RUN] run_000_f6ce5ff2 | Params: {'num_iterations': 10, 'decay_rate': 0.75}
Starting experiment 'Numerical_Experiments'. Saved in 'results/Numerical_Experiments/20260820_215201_cd04743/run_000_f6ce5ff2'
[Wrapper] [Iteration 0] -> Time 0.2325s | error_1: 0.7785 | error_2: 2.2392
[Wrapper] [Iteration 1] -> Time 0.1408s | error_1: 0.3363 | error_2: 0.9812
[Wrapper] [Iteration 2] -> Time 0.3429s | error_1: -0.1148 | error_2: -0.0777
[Wrapper] [Iteration 3] -> Time 0.0575s | error_1: 0.0530 | error_2: 0.2033
[Wrapper] [Iteration 4] -> Time 0.3113s | error_1: 0.3560 | error_2: -0.2397
[Wrapper] [Iteration 5] -> Time 0.0423s | error_1: 0.0607 | error_2: -0.2613
[Wrapper] [Iteration 6] -> Time 0.2234s | error_1: -0.2072 | error_2: 0.7887
[Wrapper] [Iteration 7] -> Time 0.0437s | error_1: 0.0458 | error_2: 0.9085
[Wrapper] [Iteration 8] -> Time 0.0764s | error_1: 0.0738 | error_2: -0.4562
[Wrapper] [Iteration 9] 

If one wants to run the experiments again, the Runner will automatically skip the succeeded cases, providing the suite ID.

In [13]:
!ls results/Numerical_Experiments/

20260820_215104_cd04743 20260820_215201_cd04743


In [14]:
# in our case will be the latest
runner = Runner(suite_name = "Numerical_Experiments", suite_id = "20260820_215201_cd04743")

results = runner.run_grid(
        func=numerical_experiment, 
        param_grid=params_grid, 
        stop_on_error=False
    )


[2026-08-20 21:52:20] Starting suite: 'Numerical_Experiments' | Total configs: 3

[SKIP] run_000_f6ce5ff2 completed in a previous session.
[RUN] run_001_dbcb01e1 | Params: {'num_iterations': -20, 'decay_rate': 1.5}
Starting experiment 'Numerical_Experiments'. Saved in 'results/Numerical_Experiments/20260820_215201_cd04743/run_001_dbcb01e1'
[ERROR] run_001_dbcb01e1 failed.
Failed in 0.0373s
[SKIP] run_002_8c8925b3 completed in a previous session.

Suite 'Numerical_Experiments' finished in 00:00:00 (0.04s).


For more edge cases and error handling, refer to the source code.